# Bab 20: Analisis Komponen Utama

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab20/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab20")
print("Folder kerja:", os.getcwd())

## Listing 20.1: PCA langkah demi langkah dan dengan scikit-learn

`kode/bab20/pca_mini.py`

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

X = np.array([[4, 5], [5, 4], [6, 6], [7, 8], [8, 7]], float)
print("rata-rata:", X.mean(0))
Xc = X - X.mean(0)
S = Xc.T @ Xc / (len(X) - 1)
print("matriks kovarians:\n", S)
nilai, vektor = np.linalg.eigh(S)
urut = np.argsort(nilai)[::-1]
nilai, vektor = nilai[urut], vektor[:, urut]
print("nilai eigen:", nilai.round(4))
print("vektor eigen (kolom):\n", vektor.round(4))
print("porsi varians:", (nilai / nilai.sum()).round(3))
Z = Xc @ vektor
print("skor komponen:\n", Z.round(3))
pca = PCA().fit(X)
print("sklearn komponen (baris):\n", pca.components_.round(4))
print("sklearn varians:", pca.explained_variance_.round(4))

## Listing 20.2: PCA dengan dan tanpa standardisasi

`kode/bab20/skala.py`

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

d = pd.read_csv("kode/data/mutu_biji.csv")
KOL = ["ketinggian_mdpl", "kadar_air", "densitas_gl",
       "ukuran_screen", "cacat_primer", "cacat_sekunder"]
X = d[KOL]
hasil = {"simp. baku": X.std()}
for nama, data in [("PC1 mentah", X),
                   ("PC1 baku", StandardScaler().fit_transform(X))]:
    pca = PCA().fit(data)
    hasil[nama] = pd.Series(pca.components_[0], KOL)
    rasio = pca.explained_variance_ratio_[:3].round(3)
    print(f"{nama:10s}: porsi varians 3 komponen pertama {rasio}")
print((pd.DataFrame(hasil).round(3) + 0.0).to_string())

## Listing 20.3: PCA sebelum regresi logistik churn

`kode/bab20/pca_praproses.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab20')
import sys; sys.path.insert(0, "kode/bab15")
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from siapkan import muat

df, X = muat()
y = df.churn
pca = make_pipeline(StandardScaler(), PCA()).fit(X)
rasio = pca[-1].explained_variance_ratio_
print("porsi varians :", rasio.round(3))
print("kumulatif     :", rasio.cumsum().round(3))

for k in [1, 2, 3, 5, 7]:
    model = make_pipeline(StandardScaler(), PCA(n_components=k),
                          LogisticRegression(max_iter=1000))
    auc = cross_val_score(model, X, y, cv=5, scoring="roc_auc")
    print(f"PCA {k} komponen -> logistik: AUC {auc.mean():.3f}")

## Listing 20.4: Studi kasus dimensi rasa kopi

`kode/bab20/studi_kasus.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

d = pd.read_csv("kode/data/profil_rasa.csv")
ATR = d.columns[3:]
X = StandardScaler().fit_transform(d[ATR])
print(f"{len(d)} kopi x {len(ATR)} atribut rasa")
R = np.corrcoef(X.T)
print(f"korelasi |r|>0,6 antaratribut: "
      f"{(np.abs(R[np.triu_indices(12, 1)]) > .6).sum()} dari 66")

pca = PCA().fit(X)
lam = pca.explained_variance_
rng = np.random.default_rng(20)
acak = np.array([PCA().fit(rng.standard_normal(X.shape))
                 .explained_variance_ for _ in range(500)])
batas = np.percentile(acak, 95, axis=0)
print("\nkomp  nilai eigen  porsi  kumulatif  batas acak 95%")
for i in range(6):
    print(f"PC{i + 1:<3d} {lam[i]:10.3f}  {lam[i] / lam.sum():.3f}"
          f"  {lam[:i + 1].sum() / lam.sum():9.3f}  {batas[i]:13.3f}")
k = int((lam > batas).sum())
print(f"komponen di atas batas acak: {k}")

pca3 = PCA(n_components=3).fit(X)
Z = pca3.transform(X)
# arah tanda: kekentalan, rasa ikutan, dan manis dibuat positif
acuan = [9, 11, 8]
arah = np.sign([np.corrcoef(X[:, j], Z[:, c])[0, 1]
                for c, j in enumerate(acuan)])
Z = Z * arah
L = pd.DataFrame([[np.corrcoef(X[:, j], Z[:, c])[0, 1]
                   for c in range(3)] for j in range(12)],
                 index=ATR, columns=["PC1", "PC2", "PC3"])
print("\nloading (korelasi atribut-komponen):")
print(L.round(2).to_string())
r2 = (L ** 2).sum(1)
print("\nR^2 atribut dari 3 komponen, min/median:",
      f"{r2.min():.2f} / {r2.median():.2f}")
print("R^2 atribut dari 2 komponen, min/median:",
      f"{(L.iloc[:, :2] ** 2).sum(1).min():.2f} / "
      f"{(L.iloc[:, :2] ** 2).sum(1).median():.2f}")

asli = pca3.components_.T
kongruen = []
for _ in range(500):
    i = rng.integers(0, len(X), len(X))
    b = PCA(n_components=3).fit(X[i]).components_.T
    kongruen.append(np.abs((asli * b).sum(0)))
kongruen = np.array(kongruen)
print("kemiripan loading bootstrap (median, persentil 5):")
for j in range(3):
    print(f"  PC{j + 1}: {np.median(kongruen[:, j]):.3f}"
          f"  {np.percentile(kongruen[:, j], 5):.3f}")

skor = pd.DataFrame(Z[:, :2], columns=["berat", "ikutan"])
skala = (1 + 4 * (skor - skor.min()) / (skor.max() - skor.min()))
hasil = pd.concat([d[["kopi", "spesies"]], skor.round(2),
                   skala.round().astype(int).add_suffix("_15")],
                  axis=1)
print("\ncontoh kartu menu (skala 1-5):")
pilih = ["Kintamani", "Kerinci", "Bajawa", "Toraja", "Gayo",
         "Mandailing", "Lampung Robusta"]
print(hasil.set_index("kopi").loc[pilih].iloc[:, 1:].to_string())

with open("gambar/bab20/skor.dat", "w") as f:
    f.write("pc1 pc2 rob nama\n")
    for _, r in hasil.iterrows():
        f.write(f"{r.berat:.3f} {r.ikutan:.3f} "
                f"{int(r.spesies == 'robusta')} {{{r.kopi}}}\n")
with open("gambar/bab20/loading.dat", "w") as f:
    f.write("l1 l2 nama\n")
    for a, r in L.iterrows():
        f.write(f"{r.PC1:.3f} {r.PC2:.3f} {{{a}}}\n")
with open("gambar/bab20/scree.dat", "w") as f:
    f.write("k lam kum acak\n")
    for i in range(12):
        f.write(f"{i + 1} {lam[i]:.3f} {lam[:i + 1].sum() / 12:.3f}"
                f" {batas[i]:.3f}\n")